# Fundamentals of Data Science — Case Study 4 (Alternative Implementation)
## Predicting Student Dropout and Academic Success

This notebook is an alternative implementation of the uploaded `Case_Study_4.ipynb`. It keeps the same overall analysis tasks while reorganizing the code and adding validation.

**Dataset required:** `Studentdata.csv` (semicolon-delimited file with a `Target` column containing outcomes such as `Graduate`, `Dropout`, and `Enrolled`).

**Sections**
1. Load and validate data
2. Exploratory analysis and outcome distribution
3. Compare age/admission/first-semester grades across outcomes
4. Run ANOVA and Kruskal–Wallis tests
5. Train and evaluate multiclass logistic regression
6. Summarize findings and download plots/results

> The notebook computes results from the dataset you upload. It does not assume or hard-code accuracy, p-values, or conclusions.


## 1. Import libraries and upload the dataset

In [ ]:
!pip -q install pandas numpy scipy scikit-learn matplotlib seaborn


In [ ]:
import os
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, balanced_accuracy_score
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
OUT_DIR = Path("/content/case_study_4_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

from google.colab import files
uploaded = files.upload()

csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Upload Studentdata.csv (or another CSV containing a Target column).")
file_path = csv_files[0]
print("Selected file:", file_path)


## 2. Load and validate the dataset
The source notebook uses `sep=';'`. This notebook checks the parsed columns and gives a clear error if the expected target column is missing.

In [ ]:
# Studentdata.csv is normally semicolon-delimited.
df = pd.read_csv(file_path, sep=";", skipinitialspace=True)
# If parsing produced only one column, try comma separation as a fallback.
if df.shape[1] == 1:
    df_comma = pd.read_csv(file_path, sep=",", skipinitialspace=True)
    if df_comma.shape[1] > 1:
        df = df_comma

df.columns = (
    df.columns.astype(str)
      .str.replace("\\t", "", regex=False)
      .str.strip()
)

if "Target" not in df.columns:
    raise ValueError(
        "Could not find a column named 'Target'. Parsed columns: "
        + ", ".join(df.columns.astype(str))
        + ". Check the file delimiter and column names."
    )

# Remove fully duplicated rows and normalize target labels.
initial_rows = len(df)
duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
df["Target"] = df["Target"].astype(str).str.strip()

# Convert numeric-looking columns while preserving non-numeric categorical columns.
for col in df.columns:
    if col != "Target":
        converted = pd.to_numeric(df[col], errors="coerce")
        # Convert only if most non-missing values can be interpreted as numbers.
        nonmissing_original = df[col].notna().sum()
        if nonmissing_original and converted.notna().sum() / nonmissing_original >= 0.90:
            df[col] = converted

print(f"Rows in file: {initial_rows:,}")
print(f"Duplicate rows removed: {duplicate_rows:,}")
print(f"Rows after de-duplication: {len(df):,}")
print(f"Number of features (including Target): {df.shape[1]}")
print(f"Missing cells: {int(df.isna().sum().sum()):,}")
print("\nColumn data types:")
display(df.dtypes.rename("dtype").to_frame())
print("\nPreview:")
display(df.head())


## 3. Exploratory data analysis

In [ ]:
# Target class counts and percentages
target_counts = df["Target"].value_counts(dropna=False)
target_percent = df["Target"].value_counts(normalize=True, dropna=False).mul(100)
target_summary = pd.DataFrame({
    "count": target_counts,
    "percent": target_percent.round(2)
})
display(target_summary)

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Target", order=target_counts.index)
plt.title("Student Outcome Class Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of students")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "target_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

# Numeric summaries
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", numeric_cols)
if numeric_cols:
    display(df[numeric_cols].describe().T)


In [ ]:
# Plot age and admission grade distributions when the columns exist
columns_to_plot = [
    ("Age at enrollment", "Age at Enrollment"),
    ("Admission grade", "Admission Grade")
]
available = [(col, title) for col, title in columns_to_plot if col in df.columns]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 4.5))
    if len(available) == 1:
        axes = [axes]
    for ax, (col, title) in zip(axes, available):
        sns.histplot(data=df, x=col, bins=25, kde=True, ax=ax)
        ax.set_title(f"{title} Distribution")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "numeric_distributions.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("Age at enrollment and Admission grade columns were not both found; skipping these plots.")


## 4. Compare numeric variables across student outcomes

In [ ]:
comparison_columns = [
    ("Admission grade", "Admission Grade by Outcome"),
    ("Curricular units 1st sem (grade)", "First-Semester Grade by Outcome")
]
found_comparisons = [(c, title) for c, title in comparison_columns if c in df.columns]

for col, title in found_comparisons:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="Target", y=col, order=target_counts.index)
    plt.title(title)
    plt.xlabel("Outcome")
    plt.ylabel(col)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{re.sub(r'[^a-z0-9]+','_',col.lower()).strip('_')}_by_target.png",
                dpi=180, bbox_inches="tight")
    plt.show()

if not found_comparisons:
    print("No expected grade columns were found; check the dataset's column names.")


## 5. Hypothesis testing: first-semester grades

**Null hypothesis (H₀):** the first-semester grade distributions/means are equal across the available outcome groups.  
**Alternative hypothesis (H₁):** at least one group differs.

ANOVA tests differences in means under assumptions; Kruskal–Wallis is a rank-based alternative. Statistical significance does not by itself establish causation or practical importance.


In [ ]:
grade_col = "Curricular units 1st sem (grade)"
alpha = 0.05
test_results = []

if grade_col not in df.columns:
    print(f"Cannot run the requested tests because '{grade_col}' is not in the dataset.")
else:
    groups = []
    group_names = []
    for label in df["Target"].dropna().unique():
        vals = pd.to_numeric(
            df.loc[df["Target"] == label, grade_col], errors="coerce"
        ).dropna().to_numpy()
        if len(vals) > 0:
            groups.append(vals)
            group_names.append(label)

    if len(groups) >= 2:
        f_stat, anova_p = stats.f_oneway(*groups)
        h_stat, kw_p = stats.kruskal(*groups)
        test_results = [
            {"test": "One-way ANOVA", "statistic": float(f_stat), "p_value": float(anova_p)},
            {"test": "Kruskal-Wallis", "statistic": float(h_stat), "p_value": float(kw_p)}
        ]
        display(pd.DataFrame(test_results))
        print("ANOVA:", "Reject H0" if anova_p < alpha else "Fail to reject H0",
              f"(alpha={alpha})")
        print("Kruskal-Wallis:", "Reject H0" if kw_p < alpha else "Fail to reject H0",
              f"(alpha={alpha})")
    else:
        print("At least two outcome groups with non-missing grades are required.")


## 6. Multiclass logistic regression

The target is `Target`; all other columns are candidate predictors. The train/test split is stratified. Imputation and scaling are fit on training data only through a scikit-learn pipeline to avoid leaking information from the test set.

This is a baseline model. If some features are only recorded after a student has already dropped out or graduated, they may leak outcome information and should be removed for a genuine early-warning model.


In [ ]:
# Separate predictors and target
X = df.drop(columns=["Target"]).copy()
y = df["Target"].astype(str)

# Encode categorical predictors with one-hot encoding; numeric columns remain numeric.
X = pd.get_dummies(X, dummy_na=True, dtype=float)

# Drop columns with no variation and ensure numeric data.
X = X.replace([np.inf, -np.inf], np.nan)
X = X.loc[:, X.nunique(dropna=False) > 1]

# Remove rows with missing target labels, if any
valid_target = y.notna() & y.ne("")
X = X.loc[valid_target]
y = y.loc[valid_target]

if y.nunique() < 2:
    raise ValueError("At least two target classes are required for classification.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"Balanced accuracy: {bal_acc:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))


In [ ]:
# Confusion matrix with model's actual class order
class_labels = list(model.named_steps["classifier"].classes_)
cm = confusion_matrix(y_test, y_pred, labels=class_labels)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_labels, yticklabels=class_labels, ax=ax
)
ax.set_title("Multiclass Logistic Regression — Confusion Matrix")
ax.set_xlabel("Predicted outcome")
ax.set_ylabel("Actual outcome")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

# Store per-class report for later reference
report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).T
display(report_df)
report_df.to_csv(OUT_DIR / "classification_report.csv")


## 7. Export results and download

In [ ]:
# Save target distribution and concise metrics
target_summary.to_csv(OUT_DIR / "target_distribution.csv")
summary = pd.DataFrame([
    {"metric": "rows_after_deduplication", "value": len(df)},
    {"metric": "feature_count_before_encoding", "value": df.shape[1] - 1},
    {"metric": "missing_cells_after_deduplication", "value": int(df.isna().sum().sum())},
    {"metric": "test_accuracy", "value": float(acc)},
    {"metric": "test_balanced_accuracy", "value": float(bal_acc)},
])
display(summary)
summary.to_csv(OUT_DIR / "model_summary.csv", index=False)

zip_file = "/content/Case_Study_4_Results.zip"
with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in OUT_DIR.rglob("*"):
        if item.is_file():
            zf.write(item, arcname=item.relative_to(OUT_DIR))

print("Output folder:", OUT_DIR)
print("Results archive:", zip_file)


In [ ]:
from google.colab import files
files.download("/content/Case_Study_4_Results.zip")
